In [1]:
# Google Colab Only
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
import warnings

import numpy as np
import torch
from maite_datasets.image_classification import MilitaryVehicles

from dataeval.data import Limit, Resize, Shuffle, View
from dataeval.extractors import TorchExtractor
from dataeval.flags import ImageStats
from dataeval.quality import Outliers

In [3]:
source = View(MilitaryVehicles("./data", image_set="test", download=True, lazy=True), [Shuffle(seed=0), Limit(300)])
source_shapes = {np.asarray(source[i][0]).shape for i in range(len(source))}
print("source dataset size:", len(source))
print("distinct source image shapes:", len(source_shapes))

source dataset size: 300
distinct source image shapes: 96


In [4]:
deployed = View(source, [Resize((224, 224))])
print("deployed image shape:", np.asarray(deployed[0][0]).shape)

deployed image shape: (3, 224, 224)


In [5]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    Outliers(flags=ImageStats.DIMENSION | ImageStats.VISUAL).evaluate(deployed)

invalidated = [w for w in caught if w.category.__name__ == "StatsInvalidatedWarning"]
print(str(invalidated[0].message))

Resize(size=(224, 224), mode='stretch', fill='mean', invalidates=None) invalidates statistics requested by Outliers: sharpness, offset_x, offset_y, width, height, size, aspect_ratio, depth, center, distance_center, distance_edge, invalid_box, rel_offset_x, rel_offset_y, rel_width, rel_height, rel_size, rel_distance_center, rel_distance_edge. These now describe the transform rather than the source data. If this is model preprocessing, move it to the extractor's transforms=; otherwise pass flags= to exclude them.


In [6]:
model = torch.nn.Sequential(torch.nn.Flatten(), torch.nn.Linear(3 * 224 * 224, 16))
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def normalize(image: torch.Tensor) -> torch.Tensor:
    """Model preprocessing: scale to [0, 1], then standardize with the ImageNet mean and standard deviation."""
    return (image / 255.0 - mean) / std


extractor = TorchExtractor(model, transforms=normalize)
embedding = extractor(np.asarray(deployed[0][0])[None])

print("view sees:", np.asarray(deployed[0][0]).shape, "-- raw pixel values")
print("extractor produces:", tuple(np.asarray(embedding).shape), "-- normalized, model-side only")

view sees: (3, 224, 224) -- raw pixel values
extractor produces: (1, 16) -- normalized, model-side only


In [7]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    Outliers(flags=ImageStats.PIXEL).evaluate(deployed)

print("warnings:", [w.category.__name__ for w in caught if "Invalidated" in w.category.__name__])

warnings: []


In [8]:
asserted = View(source, [Resize((224, 224), invalidates=ImageStats.VISUAL_SHARPNESS)])

warned = {}
for name, flags in [("DIMENSION", ImageStats.DIMENSION), ("VISUAL", ImageStats.VISUAL)]:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        Outliers(flags=flags).evaluate(asserted)
    warned[name] = any(w.category.__name__ == "StatsInvalidatedWarning" for w in caught)
    print(f"{name:>9}: {'warned' if warned[name] else 'silent'}")

DIMENSION: silent


   VISUAL: warned


In [9]:
# TEST ASSERTION CELL ###
assert len(source) == 300
assert len(source_shapes) > 90  # the text says the photographs come in many sizes
assert np.asarray(deployed[0][0]).shape == (3, 224, 224)
assert len(invalidated) == 1
assert "sharpness" in str(invalidated[0].message)
assert np.asarray(embedding).shape[-1] == 16
assert warned == {"DIMENSION": False, "VISUAL": True}